In [1]:
import sys, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.models.popularity import add_recent_counts, smoothed_ctr
from src.models.als import build_interactions, fit_indexers, add_indices, fit_als, score_als

cfg = load_config("small")
spark = get_spark(cfg)
spark.sparkContext.setCheckpointDir(str(ROOT / "data" / "checkpoints"))   # recomendado para ALS
OUT = cfg["paths"]["processed"]
RES = ROOT / "results" / "metrics"

RANKS = (16, 32, 64)
ALPHAS = (1, 10)
T_INICIO = time.time()

In [2]:
t0 = time.time()
imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))

# Candidatas de val con el CTR del baseline (igual que en los notebooks 03 y 04)
p0 = imp.filter("periodo = 'train'").agg(F.avg("label")).first()[0]
base = (imp.filter(F.col("periodo").isin("train", "val"))
           .filter(F.col("time") >= F.to_timestamp(F.lit("2019-11-13 00:00:00")))
           .select("impression_id", "user_id", "time", "news_id", "label", "periodo"))
cands = (add_recent_counts(base, hours=(24,))
         .filter("periodo = 'val'")
         .withColumn("ctr", smoothed_ctr(24, 20, p0))
         .select("impression_id", "user_id", "news_id", "label", "ctr"))

# Interacciones de entrenamiento: historiales (usuarios de train/val) + clics del 9–13 nov
inter = build_interactions(beh, imp).cache()
u_idx, i_idx = fit_indexers(inter)
n_u, n_i = len(u_idx.labelsArray[0]), len(i_idx.labelsArray[0])
inter_idx = add_indices(inter, u_idx, i_idx).select("u", "i", "r").cache()

# Candidatas indexadas; fría = usuario o noticia sin vector ALS
cands_idx = (add_indices(cands, u_idx, i_idx)
             .withColumn("fria", (F.col("u") == n_u) | (F.col("i") == n_i))
             .cache())

# Impresiones sin ninguna candidata fría (para el diagnóstico)
imp_calidas = (cands_idx.groupBy("impression_id")
                        .agg(F.max(F.col("fria").cast("int")).alias("alguna_fria"))
                        .filter("alguna_fria = 0").select("impression_id"))
cands_diag = cands_idx.join(imp_calidas, "impression_id").cache()

print(f"Interacciones: {inter_idx.count():,} | usuarios: {n_u:,} | noticias: {n_i:,}")
cands_idx.agg(F.round(F.avg(F.col("fria").cast("int")), 3).alias("pct_candidatas_frias"),
              F.round(F.sum(F.col("label") * F.col("fria").cast("int")) / F.sum("label"), 3)
               .alias("pct_clics_frios")).show()
print(f"Impresiones sin candidatas frías (diagnóstico): {cands_diag.select('impression_id').distinct().count():,} de 30,270")
print(f"Celda 2: {time.time() - t0:.0f} s")

Interacciones: 1,102,146 | usuarios: 49,906 | noticias: 38,576
+--------------------+---------------+
|pct_candidatas_frias|pct_clics_frios|
+--------------------+---------------+
|               0.571|           0.64|
+--------------------+---------------+

Impresiones sin candidatas frías (diagnóstico): 239 de 30,270
Celda 2: 116 s


In [3]:
def eval_score(df, score_col, nombre, conjunto):
    per_imp, res = evaluate(df.withColumn("score", F.col(score_col)))
    per_imp.unpersist()
    return {"modelo": nombre, "conjunto": conjunto, **res}

t0 = time.time()
filas = [
    eval_score(cands_idx, "ctr", "baseline_ctr_24h_a20", "todas"),
    eval_score(cands_diag, "ctr", "baseline_ctr_24h_a20", "sin_frias"),
]
print(f"Baseline todas: AUC = {filas[0]['auc']:.4f} | sin frías: AUC = {filas[1]['auc']:.4f} "
      f"({time.time() - t0:.0f} s)")

Baseline todas: AUC = 0.6475 | sin frías: AUC = 0.7542 (15 s)


In [4]:
for rank in RANKS:
    for alpha in ALPHAS:
        t0 = time.time()
        nombre = f"als_r{rank}_a{alpha}"
        model = fit_als(inter_idx, rank=rank, alpha=alpha)
        t_fit = time.time() - t0

        scored = score_als(model, cands_idx, cold="neutral").cache()
        filas.append(eval_score(scored, "score_als", nombre, "todas"))
        filas.append(eval_score(scored.join(imp_calidas, "impression_id"), "score_als", nombre, "sin_frias"))
        scored.unpersist()

        print(f"{nombre:13s} AUC todas = {filas[-2]['auc']:.4f} | sin frías = {filas[-1]['auc']:.4f} "
              f"(entrenar {t_fit:.0f} s, total {time.time() - t0:.0f} s)")

print(f"\nTiempo acumulado: {(time.time() - T_INICIO) / 60:.1f} min")

als_r16_a1    AUC todas = 0.5243 | sin frías = 0.5992 (entrenar 37 s, total 55 s)
als_r16_a10   AUC todas = 0.5280 | sin frías = 0.5805 (entrenar 24 s, total 41 s)
als_r32_a1    AUC todas = 0.5263 | sin frías = 0.5963 (entrenar 32 s, total 47 s)
als_r32_a10   AUC todas = 0.5301 | sin frías = 0.6267 (entrenar 29 s, total 43 s)
als_r64_a1    AUC todas = 0.5255 | sin frías = 0.5547 (entrenar 50 s, total 69 s)
als_r64_a10   AUC todas = 0.5298 | sin frías = 0.5904 (entrenar 58 s, total 81 s)

Tiempo acumulado: 7.8 min


In [5]:
mejor = max((f for f in filas if f["conjunto"] == "todas" and f["modelo"].startswith("als")),
            key=lambda f: f["auc"])
r, a = (int(x[1:]) for x in mejor["modelo"].split("_")[1:])

t0 = time.time()
model = fit_als(inter_idx, rank=r, alpha=a)
scored = score_als(model, cands_idx, cold="bottom")
filas.append(eval_score(scored, "score_als", f"{mejor['modelo']}_frias_fondo", "todas"))
print(f"{mejor['modelo']}: neutro = {mejor['auc']:.4f} | al fondo = {filas[-1]['auc']:.4f} "
      f"({time.time() - t0:.0f} s)")

als_r32_a10: neutro = 0.5301 | al fondo = 0.5037 (42 s)


In [6]:
tabla = (pd.DataFrame(filas)[["modelo", "conjunto", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
         .sort_values(["conjunto", "auc"], ascending=[True, False]).reset_index(drop=True))
(tabla.assign(periodo="val", dataset=cfg["dataset"])
      .to_csv(RES / "05_als_val.csv", index=False))
print(f"Tiempo total: {(time.time() - T_INICIO) / 60:.1f} min")
tabla.round(4)

Tiempo total: 8.5 min


,modelo,conjunto,auc,mrr,ndcg5,ndcg10,impresiones
0,baseline_ctr_24h_a20,sin_frias,0.7542,0.7424,0.7789,0.7970,239
1,als_r32_a10,sin_frias,0.6267,0.6840,0.7429,0.7508,239
2,als_r16_a1,sin_frias,0.5992,0.6680,0.7268,0.7382,239
3,als_r32_a1,sin_frias,0.5963,0.6712,0.7299,0.7390,239
4,als_r64_a10,sin_frias,0.5904,0.6636,0.7276,0.7366,239
5,als_r16_a10,sin_frias,0.5805,0.6650,0.7251,0.7367,239
6,als_r64_a1,sin_frias,0.5547,0.6466,0.7101,0.7228,239
7,baseline_ctr_24h_a20,todas,0.6475,0.2786,0.2936,0.3567,30270
8,als_r32_a10,todas,0.5301,0.2221,0.2264,0.2873,30270
9,als_r64_a10,todas,0.5298,0.2218,0.2264,0.2881,30270


In [7]:
spark.stop()